In [ ]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import numpy as np
import pandas as pd
import random

import os
from datetime import datetime 
from pytz import timezone

from tensorboardX import SummaryWriter
from scipy import sparse

from pytorchtools import EarlyStopping

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

print(torch.cuda.is_available())

In [ ]:
import easydict

args = easydict.EasyDict({
    "data" : '',
    "lr" : 1e-3,
    "wd" : 0.00, # weight_dacay (only Multi-DAE)
    "batch_size" : 500,
    "epochs" : 200,
    "total_anneal_steps" : 200000,
    "anneal_cap" : 0.2,
    "seed" : 1,
    "log_interval" : 100,
    "save" : '', 
})

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(device)

In [ ]:
def fix_seed(random_seed):
    """
    fix seed to control any randomness from a code 
    (enable stability of the experiments' results.)
    """
    torch.manual_seed(random_seed)
    torch.cuda.manual_seed(random_seed)
    #torch.cuda.manual_seed_all(random_seed)  # if use multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(random_seed)
    random.seed(random_seed)
    
fix_seed(args.seed)

In [ ]:
###############################################################################
# Load data
###############################################################################

class DataLoader():
    '''
    Load Movielens-10m datasets
    '''
    def __init__(self, path):
        self.pro_dir = os.path.join(path, 'pro_sg')
        assert os.path.exists(self.pro_dir), "Preprocessed files does not exist. Run data.py"

        self.n_items = self.load_n_items()

    def load_data(self, datatype='train'):
        if datatype == 'train':
            return self._load_train_data()
        elif datatype == 'validation':
            return self._load_tr_te_data(datatype)
        elif datatype == 'test':
            return self._load_tr_te_data(datatype)
        else:
            raise ValueError("datatype should be in [train, validation, test]")

    def load_n_items(self):
        unique_sid = list()
        with open(os.path.join(self.pro_dir, 'unique_sid.txt'), 'r') as f:
            for line in f:
                unique_sid.append(line.strip())
        n_items = len(unique_sid)
        return n_items

    def _load_train_data(self): 
        path = os.path.join(self.pro_dir, 'train.csv')
        tp = pd.read_csv(path)
        n_users = tp['uid'].max() + 1

        # binarize explicit feedback to make it implicit feedback (0/1)
        rows, cols = tp['uid'], tp['sid']
        data = sparse.csr_matrix((np.ones_like(rows),
                                (rows, cols)), dtype='float64',
                                shape=(n_users, self.n_items))
        return data

    def _load_tr_te_data(self, datatype='test'):
        tr_path = os.path.join(self.pro_dir, '{}_tr.csv'.format(datatype))
        te_path = os.path.join(self.pro_dir, '{}_te.csv'.format(datatype))  

        tp_tr = pd.read_csv(tr_path)
        tp_te = pd.read_csv(te_path)

        start_idx = min(tp_tr['uid'].min(), tp_te['uid'].min())     
        end_idx = max(tp_tr['uid'].max(), tp_tr['uid'].max())

        # binarize explicit feedback to make it implicit feedback (0/1)
        rows_tr, cols_tr = tp_tr['uid'] - start_idx, tp_tr['sid']
        rows_te, cols_te = tp_te['uid'] - start_idx, tp_te['sid']

        # held-out data: 10,000
        data_tr = sparse.csr_matrix((np.ones_like(rows_tr),
                                    (rows_tr, cols_tr)), dtype='float64', shape=(end_idx - start_idx + 1, self.n_items))
        data_te = sparse.csr_matrix((np.ones_like(rows_te),
                                    (rows_te, cols_te)), dtype='float64', shape=(end_idx - start_idx + 1, self.n_items))
        return data_tr, data_te

def get_count(tp, id):
    playcount_groupbyid = tp[[id]].groupby(id, as_index=False)
    count = playcount_groupbyid.size()
    return count
  
def filter_triplets(tp, min_uc=5, min_sc=0):
    if min_sc > 0:
        itemcount = get_count(tp, 'movieId')
        tp = tp[tp['movieId'].isin(itemcount.index[itemcount['size'] >= min_sc])]
    
    if min_uc > 0:
        usercount = get_count(tp, 'userId')
        tp = tp[tp['userId'].isin(usercount.index[usercount['size'] >= min_uc])]

    # update
    usercount, itemcount = get_count(tp, 'userId'), get_count(tp, 'movieId')
    return tp, usercount, itemcount

def split_train_test_proportion(data, test_prop=0.2):
    data_grouped_by_user = data.groupby('userId')
    tr_list, te_list = list(), list()

    np.random.seed(98765)

    for _, group in data_grouped_by_user:
        n_items_u = len(group)

        if n_items_u >= 5:
            idx = np.zeros(n_items_u, dtype='bool')
            idx[np.random.choice(n_items_u, size=int(test_prop * n_items_u), replace=False).astype('int64')] = True

            tr_list.append(group[np.logical_not(idx)])
            te_list.append(group[idx])
        
        else:
            tr_list.append(group)
        
    data_tr = pd.concat(tr_list)
    data_te = pd.concat(te_list)

    return data_tr, data_te

def numerize(tp, profile2id, show2id):
    uid = tp['userId'].apply(lambda x: profile2id[x])
    sid = tp['movieId'].apply(lambda x: show2id[x])
    return pd.DataFrame(data={'uid': uid, 'sid': sid}, columns=['uid', 'sid'])

In [ ]:
###############################################################################
# Load data
###############################################################################

loader = DataLoader(args.data)

n_items = loader.load_n_items()

# implicit feedback data (binary data)
train_data = loader.load_data('train') 
vad_data_tr, vad_data_te = loader.load_data('validation')
test_data_tr, test_data_te = loader.load_data('test')

N = train_data.shape[0]
idxlist = list(range(N))

In [ ]:
print(train_data.shape[0], vad_data_tr.shape[0], test_data_tr.shape[0])

## Build the Model

In [ ]:
class MultiVAE(nn.Module):
    """
    Container module for Multi-VAE.

    Multi-VAE : Variational Autoencoder with Multinomial Likelihood
    See Variational Autoencoders for Collaborative Filtering
    https://arxiv.org/abs/1802.05814
    """

    def __init__(self, p_dims, q_dims=None, dropout=0.5):
        super(MultiVAE, self).__init__()
        self.p_dims = p_dims
        if q_dims:
            assert q_dims[0] == p_dims[-1], "In and Out dimensions must equal to each other"
            assert q_dims[-1] == p_dims[0], "Latent dimension for p- and q- network mismatches."
            self.q_dims = q_dims
        else:
            self.q_dims = p_dims[::-1]

        # Last dimension for q- network is for mean and variance
        temp_q_dims = self.q_dims[:-1] + [self.q_dims[-1] * 2]
        self.q_layers = nn.ModuleList([nn.Linear(d_in, d_out) for 
            d_in, d_out in zip(temp_q_dims[:-1], temp_q_dims[1:])])
        self.p_layers = nn.ModuleList([nn.Linear(d_in, d_out) for
            d_in, d_out in zip(self.p_dims[:-1], self.p_dims[1:])])  
        self.drop = nn.Dropout(dropout)
        self.init_weights()

    def forward(self, input):
        mu, logvar = self.encode(input)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    def encode(self, input):
        h = F.normalize(input)
        h = self.drop(h)

        for i, layer in enumerate(self.q_layers):
            h = layer(h)
            if i != len(self.q_layers) - 1:
                h = torch.tanh(h)
            else:
                mu = h[:, :self.q_dims[-1]]
                logvar = h[:, self.q_dims[-1]:]
        return mu, logvar

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return eps.mul(std).add_(mu)
        else:
            return mu

    def decode(self, z):
        h = z 
        for i, layer in enumerate(self.p_layers):
            h = layer(h)
            if i != len(self.p_layers) - 1:
                h = torch.tanh(h)
        return h

    def init_weights(self):
        for layer in self.q_layers:
            # Xavier Initialization for weights
            size = layer.weight.size()
            fan_out = size[0]
            fan_in = size[1]
            std = np.sqrt(2.0/(fan_in + fan_out))
            layer.weight.data.normal_(0.0, std)

            # Normal Initialization for Biases
            layer.bias.data.normal_(0.0, 0.001)
        
        for layer in self.p_layers:
            # Xavier Initialization for weights
            size = layer.weight.size()
            fan_out = size[0]
            fan_in = size[1]
            std = np.sqrt(2.0/(fan_in + fan_out))
            layer.weight.data.normal_(0.0, std)

            # Normal Initialization for Biases
            layer.bias.data.normal_(0.0, 0.001)


def loss_function(recon_x, x, mu, logvar, epoch, anneal=1.0, smoothing=0.1):
    x_smoothed = x * (1 - smoothing) + (smoothing / 2)
    BCE = -torch.mean(torch.sum(F.log_softmax(recon_x, 1) * x_smoothed, -1))
    KLD = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    
    return BCE + anneal * KLD

In [ ]:
###############################################################################
# Build the model
###############################################################################

p_dims = [200, 600, n_items]
model = MultiVAE(p_dims).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=args.wd)
criterion = loss_function

In [ ]:
model

In [ ]:
###############################################################################
# Training code
###############################################################################

# TensorboardX Writer
now = datetime.now(timezone('Asia/Seoul')) #datetime.datetime.now()
DateAndTime = now.strftime('%Y-%m-%d-%H-%M-%S')
writer = SummaryWriter('Tensorboard/Mult-VAE/'+ DateAndTime)

def sparse2torch_sparse(data):
    """
    Convert scipy sparse matrix to torch sparse tensor with L2 Normalization
    This is much faster than naive use of torch.FloatTensor(data.toarray())
    https://discuss.pytorch.org/t/sparse-tensor-use-cases/22047/2
    """
    samples = data.shape[0]
    features = data.shape[1]
    coo_data = data.tocoo()
    indices = torch.LongTensor([coo_data.row, coo_data.col])
    row_norms_inv = 1 / np.sqrt(data.sum(1))
    row2val = {i : row_norms_inv[i].item() for i in range(samples)}
    values = np.array([row2val[r] for r in coo_data.row])
    t = torch.sparse.FloatTensor(indices, torch.from_numpy(values).float(), [samples, features])
    return t

def naive_sparse2tensor(data):
    return torch.FloatTensor(data.toarray())

def train(epoch):
    # Turn on training mode
    model.train()
    total_train_loss = 0.0
    train_loss = 0.0
    start_time = time.time()
    global update_count
    update_count = 0

    fix_seed(args.seed)
    np.random.shuffle(idxlist)
    
    for batch_idx, start_idx in enumerate(range(0, N, args.batch_size)):
        end_idx = min(start_idx + args.batch_size, N)
        data = train_data[idxlist[start_idx:end_idx]]
        data = naive_sparse2tensor(data).to(device)

        if args.total_anneal_steps > 0:
            anneal = min(args.anneal_cap, 
                            1. * update_count / args.total_anneal_steps)
        else:
            anneal = args.anneal_cap

        if epoch <= 200:
            progress = epoch / 200
            smoothing = 0.001 + (0.1 - 0.001) * min(progress, 1.0)
        else:
            smoothing = 0.1

        optimizer.zero_grad()
        recon_batch, mu, logvar = model(data)
        
        loss = criterion(recon_batch, data, mu, logvar, epoch, anneal, smoothing)
        loss.backward()
        train_loss += loss.item()
        # total_train_loss += loss.item()
        optimizer.step()

        update_count += 1

        if batch_idx % args.log_interval == 0 and batch_idx > 0:
            elapsed = time.time() - start_time
            print('| epoch {:3d} | {:4d}/{:4d} batches | ms/batch {:4.2f} | '
                    'loss {:4.2f}'.format(
                        epoch, batch_idx, len(range(0, N, args.batch_size)),
                        elapsed * 1000 / args.log_interval,
                        train_loss / args.log_interval))
            
            # Log loss to tensorboard
            n_iter = (epoch - 1) * len(range(0, N, args.batch_size)) + batch_idx
            writer.add_scalars('data/loss', {'train': train_loss / args.log_interval}, n_iter)

            start_time = time.time()
            train_loss = 0.0 

def evaluate(data_tr, data_te):
    # Turn on evaluation modes
    
    model.eval()
    total_loss = 0.0
    global update_count
    e_idxlist = list(range(data_tr.shape[0]))
    e_N = data_tr.shape[0]
    n100_list = []
    n50_list = []
    n20_list = []
    n10_list = []
    r100_list = []
    r50_list = []
    r20_list = []
    r10_list = []
    
    
    with torch.no_grad():
        for start_idx in range(0, e_N, args.batch_size):
            end_idx = min(start_idx + args.batch_size, N)
            data = data_tr[e_idxlist[start_idx:end_idx]]
            heldout_data = data_te[e_idxlist[start_idx:end_idx]]

            data_tensor = naive_sparse2tensor(data).to(device)

            if args.total_anneal_steps > 0:
                anneal = min(args.anneal_cap, 
                               1. * update_count / args.total_anneal_steps)
            else:
                anneal = args.anneal_cap

            
            smoothing = 0.0

            recon_batch, mu, logvar = model(data_tensor)

            loss = criterion(recon_batch, data_tensor, mu, logvar, anneal,smoothing)
            total_loss += loss.item()

            # Exclude examples from training set
            recon_batch = recon_batch.cpu().numpy()
            recon_batch[data.nonzero()] = -np.inf

            n100 = NDCG_binary_at_k_batch(recon_batch, heldout_data, 100)
            n50 = NDCG_binary_at_k_batch(recon_batch, heldout_data, 50)
            n20 = NDCG_binary_at_k_batch(recon_batch, heldout_data, 20)
            n10 = NDCG_binary_at_k_batch(recon_batch, heldout_data, 10)
            r100 = Recall_at_k_batch(recon_batch, heldout_data, 100)
            r50 = Recall_at_k_batch(recon_batch, heldout_data, 50)
            r20 = Recall_at_k_batch(recon_batch, heldout_data, 20)
            r10 = Recall_at_k_batch(recon_batch, heldout_data, 10)

            n100_list.append(n100)
            n50_list.append(n50)
            n20_list.append(n20)
            n10_list.append(n10)
            r100_list.append(r100)
            r50_list.append(r50)
            r20_list.append(r20)
            r10_list.append(r10)
    
    total_loss /= int(start_idx / args.batch_size + 1) #len(range(0, e_N, args.batch_size))
    n100_list = np.concatenate(n100_list)
    n50_list = np.concatenate(n50_list)
    n20_list = np.concatenate(n20_list)
    n10_list = np.concatenate(n10_list)
    r100_list = np.concatenate(r100_list)
    r50_list = np.concatenate(r50_list)
    r20_list = np.concatenate(r20_list)
    r10_list = np.concatenate(r10_list)

    return total_loss, np.mean(n100_list), np.mean(n50_list), np.mean(n20_list), np.mean(n10_list), np.mean(r100_list), np.mean(r50_list), np.mean(r20_list), np.mean(r10_list)

import bottleneck as bn

def NDCG_binary_at_k_batch(X_pred, heldout_batch, k=100):
    '''
    Normalized Discounted Cumulative Gain@k for binary relevance
    ASSUMPTIONS: all the 0's in heldout_data indicate 0 relevance
    '''
    batch_users = X_pred.shape[0]
    idx_topk_part = bn.argpartition(-X_pred, k, axis=1)
    topk_part = X_pred[np.arange(batch_users)[:, np.newaxis],
                       idx_topk_part[:, :k]]
    idx_part = np.argsort(-topk_part, axis=1) 

    idx_topk = idx_topk_part[np.arange(batch_users)[:, np.newaxis], idx_part]

    # build the discount template
    tp = 1. / np.log2(np.arange(2, k+2))

    DCG = (heldout_batch[np.arange(batch_users)[:, np.newaxis],
                         idx_topk].toarray() * tp).sum(axis=1)
    IDCG = np.array([(tp[:min(n, k)]).sum()
                     for n in heldout_batch.getnnz(axis=1)]) # heldout_batch.getnnz(axis=1) == heldout_data.toarray().sum(axis=1)
    return DCG[IDCG > 0.0] / IDCG[IDCG > 0.0]

def Recall_at_k_batch(X_pred, heldout_batch, k=100):
    batch_users = X_pred.shape[0]

    idx = bn.argpartition(-X_pred, k, axis=1)
    X_pred_binary = np.zeros_like(X_pred, dtype=bool)
    X_pred_binary[np.arange(batch_users)[:, np.newaxis], idx[:, :k]] = True

    X_true_binary = (heldout_batch > 0).toarray()
    tmp = (np.logical_and(X_true_binary, X_pred_binary).sum(axis=1)).astype(
        np.float32)
    denominator = np.minimum(k, X_true_binary.sum(axis=1))
    recall = tmp[denominator > 0.0] / denominator[denominator > 0.0]
    return recall

In [ ]:
###############################################################################
# Experiment
###############################################################################

best_n100 = -np.inf
update_count = 0

#early_stopping
#early_stopping = EarlyStopping(patience=5, verbose=True)

# At any point you can hit Ctrl + C to break out of training early.
try:
    final_n100 = 0
    final_n50 = 0
    final_n20 = 0
    final_n10 = 0
    final_r100 = 0
    final_r50 = 0
    final_r20 = 0
    final_r10 = 0

    fix_seed(args.seed)
    
    for epoch in range(1, args.epochs + 1):
        epoch_start_time = time.time()
        train(epoch)
        val_loss, n100, n50, n20, n10, r100, r50, r20, r10 = evaluate(vad_data_tr, vad_data_te)
        final_n100 += n100
        final_n50 += n50
        final_n20 += n20
        final_n50 += n10
        final_r100 += 100
        final_r50 += r50
        final_r20 += r20
        final_r10 += r10

        print('-' * 89)
        print('| end of epoch {:3d} | time: {:4.2f}s | valid loss {:4.2f} | '
                'n100 {:5.3f} | n50 {:5.3f} | r20 {:5.3f} | r50 {:5.3f}'.format(
                    epoch, time.time() - epoch_start_time, val_loss,
                    n100, n50, r20, r50))
        print('-' * 89)

        n_iter = epoch * len(range(0, N, args.batch_size))
        writer.add_scalars('data/loss', {'valid': val_loss}, n_iter)
        writer.add_scalar('data/n100', n100, n_iter)
        writer.add_scalar('data/r20', r20, n_iter)
        writer.add_scalar('data/r50', r50, n_iter)

        #Save the model if the n100 is the best we've seen so far.
        if n100 > best_n100:
            with open('', 'wb') as f:
                torch.save(model, f)
            print("Save the model of best n100")
            best_n100 = n100
    
    print('-' * 89)
    print('-' * 89)
    print("End of Training")

    writer.close()
     
except KeyboardInterrupt: # early stopping (Ctrl + C)
    print('-' * 89)
    print('Exiting from training early')

In [ ]:
# # Load the best saved model
with open('', 'rb') as f:
    model = torch.load(f)

In [ ]:
update_count=1

val_loss, n100, n50, n20, n10, r100, r50, r20, r100 = evaluate(vad_data_tr, vad_data_te)
print('=' * 89)
print('| End of training | val loss {:4.2f} | n100 {:5.3f} | n50 {:5.3f} | r50 {:5.3f} |'
        'r20 {:5.3f}'.format(val_loss, n100, n50, r50, r20))
print('=' * 89)

In [ ]:
#update_count=1

test_loss, n100, n50, n20, n10, r100, r50, r20, r10 = evaluate(test_data_tr, test_data_te)
print('=' * 89)
print('| End of training | test loss {:4.2f} | n100 {:5.5f} | n50 {:5.5f} | n20 {:5.5f} | n10 {:5.5f} | r100 {:5.5f} | r50 {:5.5f} | r20 {:5.5f} |'
        'r10 {:5.5f}'.format(test_loss, n100, n50, n20, n10, r100, r50, r20, r10))
print('=' * 89)